# Audit du jeu d'évaluation

Contrôle sans GPU : structure, doublons, catégories, difficulté et présence des URLs dans le corpus.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
cfg = load_config(ROOT / "config.yaml")
print(f"Projet : {ROOT}")


Projet : <PROJECT_ROOT>


In [2]:
import pandas as pd
from evaluation import load_golden_set, validate_golden_set

examples = load_golden_set(cfg.evaluation.expanded_golden_set_full_path)
with open(ROOT / "data/chunks/chunks.jsonl", encoding="utf-8") as handle:
    corpus_urls = [json.loads(line)["url"] for line in handle if line.strip()]
report = validate_golden_set(examples, corpus_urls)
pd.Series(report, name="valeur")


examples                100
answerable               90
unanswerable             10
duplicate_ids            []
duplicate_questions      []
missing_corpus_urls      []
structural_errors        []
is_valid               True
Name: valeur, dtype: object

In [3]:
rows = pd.DataFrame([{
    "id": item.example_id, "question": item.question, "category": item.category,
    "difficulty": item.difficulty, "split": item.split,
    "should_answer": item.should_answer, "urls": len(item.relevant_urls),
} for item in examples])
display(pd.crosstab(rows["category"], rows["split"], margins=True))
display(pd.crosstab(rows["difficulty"], rows["split"], margins=True))


split,challenge,development,regression,test,All
category,,,,,
admission,0,4,4,5,13
alumni,0,0,0,2,2
contact,0,1,1,0,2
cooperation,0,5,1,8,14
doctorat,0,4,0,0,4
financement,0,4,1,0,5
formation_continue,0,2,0,1,3
formation_initiale,0,8,6,0,14
gouvernance,0,1,2,1,4


split,challenge,development,regression,test,All
difficulty,,,,,
easy,0,2,7,0,9
hard,10,4,2,5,21
medium,0,34,11,25,70
All,10,40,20,30,100


In [4]:
assert report["is_valid"], report
assert len(examples) == 100
print("Jeu d'évaluation validé : 100 cas, aucun doublon et toutes les URLs attendues sont indexées.")


Jeu d'évaluation validé : 100 cas, aucun doublon et toutes les URLs attendues sont indexées.
